# Sentimental project : Raw data inspection

## Environment setup

In [1]:
import numpy as np
import pandas as pd

rs = 147

## Load and examine data

In [2]:
path = "../src/sentimental/data/raw/mental_health.csv"
df = pd.read_csv(path)
df.sample(10, random_state=rs)

,ID,statement,status
25430,25430,"I feel all the symptoms of depression, I was t...",Depression
38088,38088,a year ago today i moved to a new city the cit...,Depression
38568,38568,i need to talk to a professional but i can t b...,Depression
49661,49661,Life insurance policy/with bipolar diagnosis H...,Bipolar
43703,43703,back at the office still only day until anothe...,Normal
13603,13603,I am so tired of this body. I am so tired of t...,Suicidal
49075,49075,I feel like im against everyone else I have be...,Stress
2398,2398,have you showered?,Normal
32905,32905,wow! that is nice. let's stay for two nights.,Normal
38290,38290,a i continue to learn about myself i feel so m...,Depression


In [3]:
df.shape

(53043, 3)

In [4]:
df.isnull().sum()

ID             0
statement    362
status         0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df_no_id = df.drop("ID", axis=1)
dup_count = df_no_id.duplicated().sum()
dup_count

np.int64(1944)

In [7]:
print(f"Missing data count   : 362  ({round(36200 / df.shape[0], 2)}%)")
print(f"Duplicate data count : {dup_count} ({round(dup_count * 100 / df.shape[0], 2)}%)")

Missing data count   : 362  (0.68%)
Duplicate data count : 1944 (3.66%)


### Data quality issues

Data quality issues that are currently detected :
- ID column (drop before data split)
- Missing data in *statement* feature (0.68%)
- Exact duplicates without ID (3.66%) 

## Checking for label noise

In [8]:
df_copy = df.copy()
print(f"Raw data shape : {df_copy.shape}")

Raw data shape : (53043, 3)


In [9]:
# Drop ID column (required for duplicate detection)
df_copy = df_copy.drop("ID", axis=1)

# Drop rows with missing data (required for cleaner statistics)
df_copy = df_copy.dropna(axis=0)
print(f"No ID and missing data : {df_copy.shape}")

No ID and missing data : (52681, 2)


In [10]:
# Drop exact duplicates (required for detecting label noise)
df_copy = df_copy.drop_duplicates()
print(f"No duplicate data : {df_copy.shape}")

No duplicate data : (51093, 2)


In [11]:
dup_count = df_copy["statement"].duplicated().sum()
print(f"Conflicting labels count : {dup_count} ({round(dup_count * 100 / df_copy.shape[0], 2)}%)")

Conflicting labels count : 20 (0.04%)


## Basic data statistics

In [13]:
df_copy["status"].value_counts(normalize=True)

status
Normal                  0.313937
Depression              0.295422
Suicidal                0.208326
Anxiety                 0.070910
Bipolar                 0.048950
Stress                  0.044938
Personality disorder    0.017517
Name: proportion, dtype: float64

In [14]:
seq_len = df_copy["statement"].map(lambda x: len(x.split())).astype(np.int64)
percentiles = np.percentile(seq_len, [50, 90])

In [17]:
print("Sequence length statistics (words/tokens)")
print(f"Minimum : {seq_len.min()}")
print(f"Median  : {int(percentiles[0])}")
print(f"90th percentile : {int(percentiles[1])}")
print(f"Maximum : {seq_len.max()}")

Sequence length statistics (words/tokens)
Minimum : 1
Median  : 61
90th percentile : 278
Maximum : 6300


## Data ingestion plan

To properly fix data quality problems, our data ingestion script should perform
the following steps :
- Drop ID column : (53043, 3) -> (53043, 2)
- Drop rows with missing data : 53043 -> 52681
- Drop exact duplicates : 52681 -> 51093
- Drop rows with conflicting labels : 51093 -> 51055